In [0]:
alphacollector_transactions_history=dbutils.widgets.get("alphacollector_transactions_history")
cash_collection=dbutils.widgets.get("cash_collection")
office=dbutils.widgets.get("office")
alphacollector_claims_history=dbutils.widgets.get("alphacollector_claims_history")
client=dbutils.widgets.get("client")
payerdimension=dbutils.widgets.get("payerdimension")
paymentstype=dbutils.widgets.get("paymenttype")
paymentsdetails=dbutils.widgets.get("paymentsdetails")

In [0]:
spark.sql(
    f"""
DROP VIEW IF EXISTS pyt_temp;
"""
)

spark.sql(
    f"""
CREATE OR REPLACE TEMPORARY VIEW pyt_temp AS
SELECT 
    EntryDate, 
    TransDate, 
    TransCode, 
    TransType, 
    PaymentID, 
    BatchID, 
    OfficeExternalId, 
    ClaimNumber, 
    SUM(TransAmt) AS CollectedCash, 
    ReportingWeekEndingDate
FROM {alphacollector_transactions_history}
WHERE TransType = 'Payment' 
    AND ReportingWeekEndingDate = DATE(DATE_ADD(CURRENT_DATE(), -4))
GROUP BY 
    TransCode, 
    TransType, 
    PaymentID, 
    BatchID, 
    OfficeExternalId, 
    ClaimNumber, 
    ReportingWeekEndingDate,
    EntryDate, 
    TransDate;
"""
)

spark.sql(
    f"""
INSERT INTO {paymentstype} (
    paymenttypedescription,
    sourcesystem,
    transactiontype
)
SELECT DISTINCT 
    TransCode AS paymenttypedescription, 
    'CUBHUB'  AS sourcesystem,
    'Payment' AS transactiontype
FROM pyt_temp
WHERE TransCode NOT IN (
    SELECT paymenttypedescription
    FROM {paymentstype}
    WHERE sourcesystem = 'CUBHUB'
)
"""
)

spark.sql(
    f"""
-- Insert new payment details into dimension table
INSERT INTO {paymentsdetails} (
    batchid,
    type,
    bank,
    batchnumber,
    checkid,
    agencyid,
    depositid,
    sourcesystem
)
SELECT DISTINCT 
    pyt_temp.BatchID           AS batchid,
    NULL                       AS type,
    NULL                       AS bank,
    pyt_temp.BatchID           AS batchnumber,
    pyt_temp.PaymentID        AS checkid,
    pyt_temp.OfficeExternalID AS agencyid,
    pyt_temp.BatchID          AS depositid,
    'CUBHUB'                   AS sourcesystem
FROM pyt_temp
WHERE NOT EXISTS (
    SELECT 1
    FROM {paymentsdetails} pd
    WHERE COALESCE(pd.batchid, '')      = COALESCE(pyt_temp.BatchID, '')
      AND COALESCE(pd.checkid, '')      = COALESCE(pyt_temp.PaymentID, '')
      AND COALESCE(CAST(pd.agencyid AS STRING), '') 
          = COALESCE(CAST(pyt_temp.OfficeExternalID AS STRING), '')
      AND COALESCE(pd.depositid, '')    = COALESCE(pyt_temp.BatchID, '')
      AND COALESCE(pd.batchnumber, '')  = COALESCE(pyt_temp.BatchID, '')
      AND pd.sourcesystem = 'CUBHUB'
);
"""
)


In [0]:
spark.sql(
    f"""
INSERT INTO {cash_collection} (
    reporting_week_ending_date_key,
    posted_date_key,
    deposit_date_key,
    source_system_key,
    office_key,
    payor_key,
    client_key,
    payment_type_key,
    payment_detail_key,
    invoice_number,
    cash_collected 
)
SELECT 
    TRY_CAST(REPLACE(CAST(pyt.ReportingWeekEndingDate AS STRING), '-', '') AS INT)
        AS reporting_week_ending_date_key,

    TRY_CAST(
        CASE 
            WHEN pyt.EntryDate IS NULL OR TRIM(pyt.EntryDate) = '' THEN NULL
            WHEN INSTR(pyt.EntryDate, '/') = 0 THEN NULL
            ELSE DATE_FORMAT(TO_DATE(pyt.EntryDate, 'M/d/yyyy'), 'yyyyMMdd')
        END 
    AS INT) AS posted_date_key,

    TRY_CAST(
        CASE 
            WHEN pyt.TransDate IS NULL OR TRIM(pyt.TransDate) = '' THEN NULL
            WHEN INSTR(pyt.TransDate, '/') = 0 THEN NULL
            ELSE DATE_FORMAT(TO_DATE(pyt.TransDate, 'M/d/yyyy'), 'yyyyMMdd')
        END 
    AS INT) AS deposit_date_key,

    TRY_CAST(19 AS TINYINT) AS source_system_key,

    TRY_CAST(ofc.officekey AS INT) AS office_key,
    TRY_CAST(f.payerkey AS INT) AS payor_key,
    TRY_CAST(d.clientkey AS INT) AS client_key,
    TRY_CAST(g.paymenttypekey AS INT) AS payment_type_key,
    TRY_CAST(h.paymentdetailkey AS INT) AS payment_detail_key,

    TRY_CAST(pyt.ClaimNumber AS STRING) AS invoice_number,
    TRY_CAST(pyt.CollectedCash AS DOUBLE) AS cash_collected

FROM pyt_temp pyt

LEFT JOIN {office} ofc
    ON ofc.officenumber = pyt.OfficeExternalId

LEFT JOIN (
    SELECT *
    FROM (
        SELECT 
            ClaimNumber,
            MedicalRecordNumber,
            OfficeExternalId,
            PayerName,
            ROW_NUMBER() OVER (
                PARTITION BY ClaimNumber, OfficeExternalId 
                ORDER BY LoadDate DESC
            ) AS rnb
        FROM {alphacollector_claims_history}
    ) a
    WHERE rnb = 1
) b
    ON b.ClaimNumber = pyt.ClaimNumber
   AND b.OfficeExternalId = pyt.OfficeExternalId

LEFT JOIN (
    SELECT *
    FROM (
        SELECT 
            clientkey,
            medicalrecordnumber,
            ROW_NUMBER() OVER (
                PARTITION BY medicalrecordnumber 
                ORDER BY clientkey DESC
            ) AS rnb
        FROM {client}
        WHERE sourcesystem = 'CUBHUB'
    ) c
    WHERE rnb = 1
) d
    ON b.MedicalRecordNumber = d.medicalrecordnumber

LEFT JOIN (
    SELECT payerkey, name
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY name 
                   ORDER BY payerkey DESC
               ) AS rnb
        FROM {payerdimension}
        WHERE sourcesystemkey = 19
    ) e
    WHERE rnb = 1
) f
    ON f.name = b.PayerName

LEFT JOIN {paymentstype} g
    ON g.paymenttypedescription = pyt.TransCode
   AND g.sourcesystem = 'CUBHUB'

LEFT JOIN {paymentsdetails} h
    ON COALESCE(h.batchid, '')      = COALESCE(pyt.BatchID, '')
   AND COALESCE(h.checkid, '')      = COALESCE(pyt.PaymentID, '')
   AND COALESCE(CAST(h.agencyid AS STRING), '') 
       = COALESCE(CAST(pyt.OfficeExternalID AS STRING), '')
   AND COALESCE(h.batchnumber, '')  = COALESCE(pyt.BatchID, '')
   AND COALESCE(h.depositid, '')    = COALESCE(pyt.BatchID, '')
   AND h.sourcesystem = 'CUBHUB';
"""
)
